# 022: Async/Await — Event Loops, Cooperative Multitasking, and Concurrent Engine Design

## 1. THEORY LAYER

### Event Loops and Cooperative Multitasking
Traditional multitasking utilizes **preemptive scheduling** managed by the operating system, which context-switches threads arbitrarily. This introduces significant thread context-switch overhead and race conditions.

**Cooperative Multitasking** hands control back to the application. In this paradigm, tasks yield control voluntarily back to a central scheduler (the **Event Loop**) when waiting for I/O operations to complete. This allows a single OS thread to handle thousands of concurrent network connections.

### Coroutines vs Generators
- **Generators**: Produce a sequence of values over time using `yield`.
- **Coroutines**: Consume values and yield execution control back to the caller, mapping to CPython's `async def` declarations.

---

## 2. CPYTHON INTERNAL LAYER

### Generators under the hood
When an `async def` function is called, it does not execute. Instead, it returns a coroutine object. Under the hood, this coroutine is a special type of generator.
- The event loop advances execution by calling the coroutine's `.send(None)` method.
- When the coroutine reaches an `await future` statement, it suspends execution and yields control back to the loop.
- Once the I/O event completes, the loop resumes the coroutine by calling `.send(result)`.

---


In [ ]:
import sys
import time
import asyncio
from collections import deque

print("=" * 70)
print("SECTION 1: Generator Send/Receive Mechanism")
print("=" * 70)

def simple_coroutine():
    print("-> Coroutine Started")
    x = yield "Yielded Value"
    print(f"-> Coroutine Resumed with sent value: {x}")

coro = simple_coroutine()
# Prime the generator (move to first yield)
first_yield = next(coro)
print(f"Received from yield: {first_yield}")

# Resume and send value
try:
    coro.send(42)
except StopIteration:
    print("-> Coroutine Completed execution")



---
## 3. IMPLEMENTATION LAYER

### Level 1: Custom Event Loop from Scratch in Pure Python
To show how `asyncio` works under the hood, we build a functioning Cooperative Event Loop from scratch using simple generator queues.


In [ ]:
class Task:
    """Represents a scheduled unit of execution."""
    def __init__(self, coro):
        self.coro = coro
        self.value = None

class CustomEventLoop:
    """Pure Python Cooperative Event Loop Engine."""
    def __init__(self):
        self.task_queue = deque()

    def create_task(self, coro):
        task = Task(coro)
        self.task_queue.append(task)
        return task

    def run_until_complete(self):
        """Processes scheduled tasks sequentially."""
        while self.task_queue:
            task = self.task_queue.popleft()
            try:
                # Advance coroutine execution
                yielded = task.coro.send(task.value)
                
                if isinstance(yielded, float):
                    # Simulate non-blocking sleep by yielding back to queue
                    print(f"Task yielded for {yielded}s, rescheduling...")
                    self.task_queue.append(task)
                else:
                    self.task_queue.append(task)
            except StopIteration:
                # Coroutine finished executing
                pass

# Custom non-blocking sleep simulation
def async_sleep(seconds):
    yield seconds

def task_one():
    print("Task One: Part A")
    yield from async_sleep(0.1)
    print("Task One: Part B")

def task_two():
    print("Task Two: Part A")
    yield from async_sleep(0.2)
    print("Task Two: Part B")

print("\n=== Level 1: Custom Event Loop ===")
loop = CustomEventLoop()
loop.create_task(task_one())
loop.create_task(task_two())
loop.run_until_complete()



---
### Level 3: Advanced Usage Systems — Concurrent Downloader
Using Python's native `asyncio` library, we simulate a production-grade concurrent web page downloader displaying non-blocking concurrency.


In [ ]:
async def download_page(page_id, delay):
    print(f"Starting download of Page {page_id}...")
    await asyncio.sleep(delay)  # Yield control to event loop
    print(f"Finished download of Page {page_id}!")
    return f"Data from Page {page_id}"

async def main_downloader():
    # Schedule three downloads concurrently
    t0 = time.perf_counter()
    results = await asyncio.gather(
        download_page(1, 0.3),
        download_page(2, 0.1),
        download_page(3, 0.2)
    )
    t_elapsed = (time.perf_counter() - t0)
    print(f"\nAll downloads complete in {t_elapsed:.2f} seconds.")
    print(f"Results: {results}")

print("\n=== Level 3: Concurrent Downloader Simulation ===")
# Run the downloader using native asyncio loop
asyncio.run(main_downloader())



---
## 5. EXPERIMENTATION LAYER
### Blocking vs Non-Blocking Concurrency Benchmarks
We measure the execution difference between sequential blocking tasks and concurrent non-blocking tasks.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 5: Concurrency Benchmarks")
print("=" * 70)

# Blocking simulation
def blocking_task(delay):
    time.sleep(delay)  # Blocks the entire OS thread

t0 = time.perf_counter()
blocking_task(0.1)
blocking_task(0.1)
t_blocking = (time.perf_counter() - t0) * 1000

# Concurrent simulation
async def concurrent_task(delay):
    await asyncio.sleep(delay)

async def run_concurrent():
    await asyncio.gather(concurrent_task(0.1), concurrent_task(0.1))

t0 = time.perf_counter()
asyncio.run(run_concurrent())
t_concurrent = (time.perf_counter() - t0) * 1000

print(f"Blocking sequential time: {t_blocking:.2f} ms")
print(f"Concurrent loop time:    {t_concurrent:.2f} ms")



---
## 6. VISUALIZATION LAYER
```
Async Event Loop Call Stack Flow:
 
 +-------------------------+
 |     Event Loop Queue    | <----+
 +-------------------------+      | (Reschedule if not done)
   | (Pop task)                     |
   v                              |
 [ Coroutine execution ]          |
   |                              |
   +--- (await / yield I/O) ------+
```
---
## 7. INTERVIEW MASTER LAYER

### 100 Scenario-Based Async/Await Interview Q&As (Summary Selection)

1. **How does `asyncio` achieve concurrency on a single OS thread?**
   - By using cooperative multitasking. When a coroutine awaits an I/O event, it yields control back to the event loop. The loop uses non-blocking sockets (via selectors like `select` or `epoll`) to monitor other channels, executing ready coroutines in the meantime.
2. **What is the difference between `asyncio.gather` and `asyncio.wait`?**
   - `gather` accepts high-level awaitables, schedules them as tasks, and returns results in a sorted list. `wait` accepts tasks, offers lower-level control, and returns two sets: `(done, pending)`.
3. **What happens if you execute a blocking call inside an async function?**
   - It blocks the single OS thread running the event loop. No other coroutines can progress during the blocking call, defeating the purpose of asynchronous programming.
4. **How do you run CPU-bound work inside an async application?**
   - Execute the CPU-bound task in a separate thread or process pool using `loop.run_in_executor()`.
5. **What is a Future in asyncio?**
   - A Future is a low-level object representing an eventual result of an asynchronous operation. A Task is a subclass of Future that wraps and runs a coroutine.
